# Notebook 3: Random Forests — Wisdom of the Crowd

**Series:** Random Forests & Isolation Forests for Full-Stack Engineers  
**Prerequisites:** Notebooks 1 & 2  
**Author:** [Farty Bobo](https://fartybobo.com)
**What you'll build:** A Random Forest classifier and intuition for why it works

---

## The Problem with a Single Decision Tree

In Notebook 1, you trained one decision tree. It had a problem: **high variance**.

Change a few training samples → you might get a completely different tree. The tree is "opinionated" — it commits hard to the specific data it saw.

Here's a concrete example: imagine training 5 different trees on 5 slightly different samples of the same dataset. Each tree makes different splits, produces different boundaries, and if you ask them all to classify the same new flower, you might get disagreement:

- Tree 1: `versicolor`
- Tree 2: `versicolor`  
- Tree 3: `virginica`
- Tree 4: `versicolor`
- Tree 5: `virginica`

Final answer: **majority vote** → `versicolor` (3 vs 2). That's a Random Forest.

---

## The Software Engineering Analogy: Distributed Consensus

You've heard of consensus algorithms like Raft or Paxos? Multiple nodes each have a "vote" on the answer, and you take the majority. Any single node might have stale data or be wrong, but if enough nodes agree, you get a reliable result.

Random Forest is the same idea:
- Each tree = one node/replica, with slightly different data
- Each tree votes on the classification
- Majority vote = final prediction
- Or: average probabilities for a softer, continuous output

This is called an **ensemble method** — combining many weak learners into one strong learner.

---

## How Random Forest Creates Diversity

The key insight: for majority vote to outperform any single voter, the voters must make **different** errors. If all 100 trees fail on the same inputs, the ensemble is no better.

Random Forest uses two tricks to force diversity:

### 1. Bootstrap Sampling (Bagging)

Each tree is trained on a **bootstrap sample** — a sample of the training data taken *with replacement*, same size as the original.

"With replacement" means: draw a sample, put it back, draw again. You might pick the same data point multiple times, and some data points might not be picked at all.

```python
# Bootstrap sampling (simplified)
original_data = list(range(10))  # 10 samples
bootstrap     = random.choices(original_data, k=10)  # sample 10 WITH replacement
# might give: [3, 7, 3, 1, 9, 2, 3, 8, 0, 5]  -- 3 appears 3 times, 4/6 never appear!
```

Each tree sees a different scrambled version of the dataset. Some samples get overrepresented, some are entirely absent. This makes each tree biased in different directions.

The ~37% of samples never selected for a tree are called **out-of-bag (OOB)** samples — a free, built-in validation set for that tree.

### 2. Random Feature Subsets

At each node split, the tree considers only a **random subset of features**, not all of them.

If you have 100 features, each split might only consider sqrt(100) = 10 random features. This forces the trees to find creative, diverse paths through the data.

Without this, every tree would likely make the same first split (the single most informative feature), leading to very similar trees — and similar errors.

Think of it like code review: you want reviewers with different backgrounds (frontend, backend, security) each looking at a different slice of the code — not all 10 reviewers looking at the same obvious bug.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.datasets import load_iris, make_moons, make_circles, make_classification, make_blobs
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pal = sns.color_palette('colorblind')
sns.set_palette(pal)

iris = load_iris()
print('Setup complete.')

In [ ]:
# --- Demonstrate bootstrap sampling ---
# Show how each tree in a forest sees different data

np.random.seed(42)
n_samples = 20
original_indices = np.arange(n_samples)

print(f'Original dataset:  {original_indices.tolist()}')
print()
print('Bootstrap samples for 5 trees:')

for tree_num in range(5):
    bootstrap = np.random.choice(original_indices, size=n_samples, replace=True)
    bootstrap_sorted = sorted(bootstrap)
    oob = sorted(set(original_indices) - set(bootstrap))  # out-of-bag
    print(f'  Tree {tree_num+1}: {bootstrap_sorted}')
    print(f'         OOB (never seen): {oob}')
    print()

## Single Tree vs Random Forest — Decision Boundaries

The best way to understand why Random Forests work is to visually compare the decision boundaries of:
- A single decision tree
- A random forest

The **decision boundary** is the line (or surface) that separates one class from another in feature space. Think of it as the if/else logic made visual.

We'll test on several synthetic 2D datasets so we can visualize the boundary easily. Real datasets have many more dimensions, but the principle is the same.

In [ ]:
# Visualize decision boundaries for various 2D datasets.
# The colored background = model's predicted class for any point in that region.
# Darker = more confident. Lighter = uncertain.

from matplotlib.colors import ListedColormap

def plot_decision_boundaries(datasets, dataset_names, models, model_names, n_estimators=50):
    h = 0.02  # mesh step size — smaller = higher resolution, slower
    
    n_rows = len(datasets)
    n_cols = 1 + len(models)
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows), dpi=100)
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    cm = plt.cm.RdBu  # red-blue colormap for binary classification
    cm_bright = ListedColormap(['#FF4444', '#4444FF'])
    
    for row, (ds, ds_name) in enumerate(zip(datasets, dataset_names)):
        X, y = ds
        X = StandardScaler().fit_transform(X)  # normalize for visual comparison
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
        
        x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
        y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
        xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
        
        # Column 0: Raw data
        ax = axes[row, 0]
        ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap=cm_bright, edgecolors='k', s=20, lw=0.5)
        ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, edgecolors='k', s=20, alpha=0.3, lw=0.5)
        ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
        ax.set_xticks(()); ax.set_yticks(())
        if row == 0: ax.set_title('Training Data', fontsize=11, fontweight='bold')
        ax.set_ylabel(ds_name, fontsize=10)
        
        # Columns 1+: Model boundaries
        for col, (model, model_name) in enumerate(zip(models, model_names), start=1):
            ax = axes[row, col]
            
            model.fit(X_train, y_train)
            score = model.score(X_test, y_test)
            
            Z = model.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
            Z = Z.reshape(xx.shape)
            ax.contourf(xx, yy, Z, cmap=cm, alpha=0.8)
            ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap=cm_bright, edgecolors='k', s=20, lw=0.5)
            ax.set_xlim(x_min, x_max); ax.set_ylim(y_min, y_max)
            ax.set_xticks(()); ax.set_yticks(())
            if row == 0: ax.set_title(model_name, fontsize=11, fontweight='bold')
            ax.text(x_max - 0.1, y_min + 0.1, f'{score:.0%}',
                    size=12, ha='right', va='bottom',
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.suptitle('Decision Boundaries: Single Tree vs Random Forest\n(% = test accuracy, shown in corner)', 
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


# Create several 2D datasets with different geometries
np.random.seed(42)
n_inliers = 200

datasets = [
    make_moons(n_samples=n_inliers, noise=0.2, random_state=42),      # crescent shapes
    make_circles(n_samples=n_inliers, noise=0.15, factor=0.5, random_state=42),  # concentric circles
    make_blobs(n_samples=n_inliers, centers=2, cluster_std=1.5, random_state=42),  # blobs
]
dataset_names = ['Moons', 'Circles', 'Blobs']

models = [
    DecisionTreeClassifier(max_depth=5, random_state=42),
    RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42),
]
model_names = [
    'Single Decision Tree\n(max_depth=5)',
    'Random Forest\n(50 trees, max_depth=5)',
]

plot_decision_boundaries(datasets, dataset_names, models, model_names)

### What to Notice in the Plots

- **Single tree boundary:** Jagged, rectangular cutouts. The tree makes sharp axis-aligned splits. On complex shapes (moons, circles) it creates rough approximations.
- **Random Forest boundary:** Smoother curves. Many trees with slightly different splits average out into a more continuous boundary.
- **Uncertainty (lighter areas):** The light-colored regions near boundaries are where the forest is uncertain — roughly equal numbers of trees vote each way.
- **The concentric circles problem:** Notice both struggle here. These datasets require circular boundaries; axis-aligned trees naturally make rectangular cuts.

---

## Key Parameters and What They Control

| Parameter | Default | What it controls | Too low | Too high |
|---|---|---|---|---|
| `n_estimators` | 100 | Number of trees | Noisy predictions, high variance | Just slower (more is usually better) |
| `max_depth` | None | Max levels per tree | Underfitting (misses patterns) | Overfitting |
| `max_features` | `sqrt(n)` | Features considered per split | Underfitting, too random | Trees too similar, defeats diversity |
| `min_samples_leaf` | 1 | Min samples in leaf node | Overfitting (tiny leaves) | Underfitting (coarse decisions) |
| `min_samples_split` | 2 | Min samples to attempt a split | Overfitting | Underfitting |
| `max_leaf_nodes` | None | Max leaf nodes per tree | Underfitting | Overfitting |

The most important lesson: **smaller trees generalize better**. A shallow forest of many small trees often beats one big tree.

In [ ]:
# Demonstrate the effect of n_estimators: more trees = more stable predictions
# We'll train forests with 1, 5, 10, 50, 100, 200 trees and compare test accuracy

X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target
)

n_estimator_values = [1, 2, 5, 10, 20, 50, 100, 200]
n_trials = 10  # run each n_estimators 10 times to show variance

results = {n: [] for n in n_estimator_values}

for n_est in n_estimator_values:
    for seed in range(n_trials):
        rf = RandomForestClassifier(n_estimators=n_est, random_state=seed, n_jobs=-1)
        rf.fit(X_train, y_train)
        results[n_est].append(rf.score(X_test, y_test))

# Plot mean and variance
fig, ax = plt.subplots(figsize=(9, 5), dpi=100)

means = [np.mean(results[n]) for n in n_estimator_values]
stds  = [np.std(results[n]) for n in n_estimator_values]

ax.errorbar(n_estimator_values, means, yerr=stds, marker='o', capsize=5, 
            color=pal.as_hex()[0], linewidth=2, markersize=7)
ax.fill_between(n_estimator_values,
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                alpha=0.2, color=pal.as_hex()[0])

ax.set_xscale('log')
ax.set_xlabel('n_estimators (number of trees)', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('More Trees = More Stable Accuracy\n(error bars show variance across 10 runs)', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_xticks(n_estimator_values)
ax.set_xticklabels(n_estimator_values)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\nInsights:')
print(f'  n_estimators=1  (single tree):  mean={means[0]:.1%}, std={stds[0]:.1%} <- HIGH variance')
print(f'  n_estimators=100:               mean={means[-2]:.1%}, std={stds[-2]:.1%} <- much more stable')

In [ ]:
# Demonstrate max_depth: deeper trees overfit, shallower trees generalize

depths = [1, 2, 3, 5, 10, 20, None]  # None = unlimited
depth_labels = [str(d) if d is not None else '∞' for d in depths]

train_scores = []
test_scores  = []

for depth in depths:
    rf = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    train_scores.append(rf.score(X_train, y_train))
    test_scores.append(rf.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(9, 5), dpi=100)
x = np.arange(len(depths))
width = 0.35

bars1 = ax.bar(x - width/2, train_scores, width, label='Train accuracy', color=pal.as_hex()[0], alpha=0.8)
bars2 = ax.bar(x + width/2, test_scores, width, label='Test accuracy', color=pal.as_hex()[1], alpha=0.8)

ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Overfitting: Train Accuracy vs Test Accuracy by max_depth', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(depth_labels)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.set_ylim(0.5, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Annotate the gap between train and test
for i, (tr, te) in enumerate(zip(train_scores, test_scores)):
    gap = tr - te
    if gap > 0.02:
        ax.annotate(f'gap: {gap:.0%}', xy=(i, te), xytext=(i, te - 0.05),
                    fontsize=8, ha='center', color='red')

plt.tight_layout()
plt.show()

### Reading the Overfitting Chart

- **Large gap between train and test accuracy** = overfitting. The model memorized training data but doesn't generalize.
- **Small gap** = good. Similar performance on unseen data.
- `max_depth=∞` → train=100%, test drops. Classic overfit.
- There's a "sweet spot" depth that maximizes test accuracy — finding it is called **hyperparameter tuning** (Notebook topic next).

---

## Probability Output: Confidence Scores

One of the most useful features of Random Forests: they output **class probabilities**, not just a label.

`predict_proba()` returns the fraction of trees that voted for each class. This gives you a confidence score — essential for:
- Setting decision thresholds (`predict if confidence > 0.8`)
- Flagging uncertain predictions for human review
- Ranking predictions (e.g., top-k recommendations)

In [ ]:
# Train a Random Forest on Iris and examine probability outputs

rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf.fit(X_train, y_train)

# Get predictions and probabilities on the test set
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)  # shape: (n_test, n_classes)

# Find the "most confident" and "least confident" predictions
confidence = y_proba.max(axis=1)  # confidence = probability of predicted class

print('Overall test accuracy:', f"{rf.score(X_test, y_test):.1%}")
print()
print('Most confident predictions (top 5):')
top_idx = np.argsort(confidence)[::-1][:5]
for i in top_idx:
    probs = y_proba[i]
    true_label = iris.target_names[y_test[i]]
    pred_label = iris.target_names[y_pred[i]]
    correct = '✓' if y_test[i] == y_pred[i] else '✗'
    print(f'  {correct} pred={pred_label:12s} true={true_label:12s}  {[f"{p:.0%}" for p in probs]}')

print()
print('Least confident predictions (bottom 5):')
bottom_idx = np.argsort(confidence)[:5]
for i in bottom_idx:
    probs = y_proba[i]
    true_label = iris.target_names[y_test[i]]
    pred_label = iris.target_names[y_pred[i]]
    correct = '✓' if y_test[i] == y_pred[i] else '✗'
    print(f'  {correct} pred={pred_label:12s} true={true_label:12s}  {[f"{p:.0%}" for p in probs]}')

---

## Handling Unbalanced Classes

Often your classes aren't equally represented. Example: fraud detection — maybe 1% of transactions are fraudulent. If a model just says "not fraud" for everything, it gets 99% accuracy but is completely useless.

Random Forest handles this with the `class_weight` parameter:
- `class_weight='balanced'` — automatically weights classes inversely proportional to their frequency
- `class_weight={0: 1, 1: 10}` — manually give class 1 ten times more weight

This changes the Gini impurity calculation to penalize misclassifying rare classes more heavily.

```python
# For a dataset with 99% class 0 and 1% class 1:
rf_balanced = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',  # or {0: 1, 1: 99}
    random_state=42
)
```

---

## Summary

| Concept | What it is | Key insight |
|---|---|---|
| **Ensemble method** | Combining many weak classifiers | Individual errors cancel out across the forest |
| **Bootstrap sampling** | Training each tree on resampled data with replacement | Forces diversity between trees |
| **Random feature subsets** | Each split considers only a random subset of features | Prevents all trees from making the same splits |
| **n_estimators** | Number of trees | More = more stable, diminishing returns after ~100 |
| **max_depth** | Max depth per tree | Key overfitting control |
| **predict_proba** | Fraction of trees voting for each class | More useful than hard labels — gives confidence |
| **class_weight** | Weight given to each class in Gini calc | Fix unbalanced datasets |

---

## What's Next

**Notebook 4: NULL Handling & Feature Importance** — Real data has gaps (NULLs). We'll cover strategies for dealing with them. Then we'll learn how to ask the trained forest: "Which features actually mattered most?"